In [1]:
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
minute_data = pd.read_csv('datasets/SPY-minute-aggregations(in).csv', parse_dates=['Timestamp'])

In [3]:
minute_data.head()

,Timestamp,Open,High,Low,Close,Volume
0,2020-10-19 08:00:00+00:00,348.60,348.93,348.60,348.93,16043
1,2020-10-19 08:01:00+00:00,348.98,349.12,348.98,349.05,6530
2,2020-10-19 08:02:00+00:00,349.09,349.09,349.07,349.07,1633
3,2020-10-19 08:05:00+00:00,348.85,348.85,348.85,348.85,250
4,2020-10-19 08:06:00+00:00,348.80,348.80,348.62,348.62,1749


In [4]:
import plotly.graph_objs as go


In [5]:
fig = go.Figure(data=go.Scatter(x=minute_data[:1000]['Timestamp'], 
                                    y=minute_data[:1000]['Close'], 
                                    mode='lines'))
fig.update_layout(title='SPY Minute Close Prices')
fig.show()

In [7]:
def get_velocity(time):
    price = minute_data['Close'][time.index]
    return price.diff() / time.diff().dt.total_seconds()

velocity = get_velocity(minute_data['Timestamp'])


In [9]:
print(velocity)

0              NaN
1         0.002000
2         0.000333
3        -0.001222
4        -0.003833
            ...   
299995   -0.001333
299996    0.000500
299997    0.005000
299998   -0.000667
299999   -0.001167
Length: 300000, dtype: float64


In [16]:
def price_kinematics(df, price_col="Close", time_col="Timestamp"):
    """
    Computes time-aware derivatives and percent-change measures:
      d1   : first derivative (slope of price per second)
      d2   : second derivative (change in slope per second)
      pct  : percent change between points (return)
      dpct : change in percent change (Δ return)
    """
    out = df.copy()
    out[time_col] = pd.to_datetime(out[time_col], utc=True, errors="coerce")
    out = out.sort_values(time_col).reset_index(drop=True)

    # Δt in seconds (handle uneven spacing)
    dt = out[time_col].diff().dt.total_seconds()
    med_dt = np.nanmedian(dt)
    dt.iloc[0] = med_dt if np.isfinite(med_dt) and med_dt > 0 else 1.0

    # first derivative (slope)
    out["d1"] = out[price_col].diff() / dt

    # second derivative (change of slope)
    out["d2"] = out["d1"].diff() / dt

    # percent change between points (simple return)
    out["pct"] = out[price_col].pct_change()

    # change in percent change (Δ return)
    out["dpct"] = out["pct"].diff()

    return out

In [18]:
result = price_kinematics(minute_data, price_col="Close", time_col="Timestamp")

# peek
print(result.head(10))

                  Timestamp    Open    High     Low   Close  Volume        d1  \
0 2020-10-19 08:00:00+00:00  348.60  348.93  348.60  348.93   16043       NaN   
1 2020-10-19 08:01:00+00:00  348.98  349.12  348.98  349.05    6530  0.002000   
2 2020-10-19 08:02:00+00:00  349.09  349.09  349.07  349.07    1633  0.000333   
3 2020-10-19 08:05:00+00:00  348.85  348.85  348.85  348.85     250 -0.001222   
4 2020-10-19 08:06:00+00:00  348.80  348.80  348.62  348.62    1749 -0.003833   
5 2020-10-19 08:07:00+00:00  348.58  348.62  348.56  348.62    1872  0.000000   
6 2020-10-19 08:08:00+00:00  348.70  348.70  348.70  348.70     184  0.001333   
7 2020-10-19 08:09:00+00:00  348.73  348.73  348.72  348.72     200  0.000333   
8 2020-10-19 08:10:00+00:00  348.86  348.86  348.86  348.86     335  0.002333   
9 2020-10-19 08:11:00+00:00  348.84  348.84  348.84  348.84     356 -0.000333   

         d2       pct      dpct  
0       NaN       NaN       NaN  
1       NaN  0.000344       NaN  
2 -0.0

/var/folders/5h/lv_gm99s337dww6tj437kv080000gn/T/ipykernel_1473/3280460073.py:16: SettingWithCopyWarning:

modifications to a method of a datetimelike object are not supported and are discarded. Change values on the original.

